<a href="https://colab.research.google.com/github/Munjiwon/SpecialTopics-in-TextMining/blob/master/ch02/ngram_smoothing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2주차 실습 3 — n-gram 언어 모델과 평활화

**이 노트북의 새 개념**: 최대우도 바이그램의 영확률 문제와 Add-k · Kneser-Ney 평활화의 차이.

- 최대우도: p̂(w_t | w_{t−1}) = c(w_{t−1}, w_t) / c(w_{t−1})
- Kneser-Ney 연속 확률: p_KN(w) ∝ |{v : c(v, w) > 0}|

In [1]:
import sys
print("Python", sys.version.split()[0])

Python 3.13.15


In [2]:
# 말뭉치 올리기 — 1주차와 같은 파일을 쓴다(빈 줄로 구분된 문단 하나를 문서 하나로 본다)
CORPUS_PATH = "/content/corpus.txt"      # Colab 밖에서 실행할 때는 이 경로를 직접 바꾼다
try:
    from google.colab import files
    uploaded = files.upload()
    CORPUS_PATH = "/content/" + next(iter(uploaded))
except ImportError:
    pass

with open(CORPUS_PATH, encoding="utf-8") as f:
    raw = f.read()
docs = [d.strip() for d in raw.split("\n\n") if len(d.strip()) > 20]   # 문단 = 문서
print(f"문서 {len(docs):,}개, 예시: {docs[0][:60]}...")

Saving corpus.txt to corpus.txt
문서 1개, 예시: 0	노래가 너무 적음
0	돌겠네 진짜. 황숙아, 어크 공장 그만 돌려라. 죽는다.
1	막노동 체험판 막노동 ...


In [3]:
import math
from collections import Counter, defaultdict

tokens = raw.split()
cut = int(len(tokens) * 0.9)
train, test = tokens[:cut], tokens[cut:]
uni = Counter(train)
bi = Counter(zip(train, train[1:]))
V = len(uni)
test_pairs = list(zip(test, test[1:]))
zero = sum(1 for p in test_pairs if bi[p] == 0)
print(f"평가 바이그램 {len(test_pairs):,}개 중 학습에 없던 것 {zero:,}개 ({zero/len(test_pairs):.1%}) → 최대우도 확률 0")

평가 바이그램 116,610개 중 학습에 없던 것 85,471개 (73.3%) → 최대우도 확률 0


## Add-k 평활화
모든 칸에 k 를 더한다. 관측 안 된 방대한 조합에 확률을 균등하게 나눠 줘 성능이 나쁘다.

In [4]:
def p_addk(prev, w, k=0.1):
    return (bi[(prev, w)] + k) / (uni[prev] + k * (V + 1))     # +1 은 미등록어 한 칸

def ppl(prob):
    s = sum(math.log2(prob(a, b)) for a, b in test_pairs)
    return 2 ** (-s / len(test_pairs))

for k in [1.0, 0.1, 0.01]:
    print(f"Add-{k:<5} PPL = {ppl(lambda a, b: p_addk(a, b, k)):,.0f}")

Add-1.0   PPL = 145,881
Add-0.1   PPL = 91,657
Add-0.01  PPL = 64,136


## 보간 Kneser-Ney
관측 카운트에서 D 를 빼고(절대 할인), 그 질량을 "몇 종류의 문맥 뒤에 나타났는가"로 정한 연속 확률에 나눠 준다.

In [5]:
D = 0.75
followers = defaultdict(set)        # prev 뒤에 나온 서로 다른 단어들
preceders = defaultdict(set)        # w 앞에 나온 서로 다른 단어들
for a, b in bi:
    followers[a].add(b); preceders[b].add(a)
n_bigram_types = len(bi)

def p_cont(w):
    # 연속 확률 — 미등록어는 1 / (바이그램 종류 수 + 1) 로 작게 둔다
    return (len(preceders[w]) or 1) / (n_bigram_types + 1)

def p_kn(prev, w):
    c_prev = uni[prev]
    if c_prev == 0:
        return p_cont(w)
    lam = D * len(followers[prev]) / c_prev          # 할인으로 모은 질량
    return max(bi[(prev, w)] - D, 0) / c_prev + lam * p_cont(w)

print(f"Kneser-Ney PPL = {ppl(p_kn):,.0f}")
print("연속 확률이 큰 단어:", sorted(uni, key=lambda w: -len(preceders[w]))[:10])

Kneser-Ney PPL = 12,735
연속 확률이 큰 단어: ['1', '0', '게임', '너무', '이', '그냥', '좀', '더', '다', '정말']


## 직접 해 보기
1. D 를 0.5, 0.9 로 바꾸면 PPL 이 어떻게 변하는가?
2. 빈도는 높은데 앞 문맥 종류가 적은 단어를 찾아 p_cont 가 작게 나오는지 확인해 보자(예: 고유명사의 뒷부분).
3. 형태소 토큰으로 바꿔 같은 실험을 하면 영확률 비율이 어떻게 달라지는가?